In [ ]:
!pip install --upgrade petl

In [ ]:
import requests
import petl as etl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import time

In [ ]:
def fetch_bitcoin_price():
    url = "https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd"
    response = requests.get(url)
    data = response.json()
    if 'bitcoin' in data and 'usd' in data['bitcoin']:
        price = data['bitcoin']['usd']
    else:
        print("Error: Unexpected API response format")  
        price = None  
    return price

In [ ]:
def store_bitcoin_price(price, timestamp):
    table = etl.fromdicts([{'timestamp': timestamp, 'price': price}])  
    return table

In [ ]:
def time_series_analysis(dataframe):
    dataframe['timestamp'] = pd.to_datetime(dataframe['timestamp'])
    
    dataframe.set_index('timestamp', inplace=True)
    
    dataframe['moving_avg'] = dataframe['price'].rolling(window=7).mean()

    dataframe['volatility'] = dataframe['price'].rolling(window=7).std()
    
    return dataframe

In [ ]:
def visualize_data(dataframe):
    plt.figure(figsize=(12, 6))
    
    sns.lineplot(data=dataframe, x='timestamp', y='price', label='Bitcoin Price (USD)', color='blue')
    sns.lineplot(data=dataframe, x='timestamp', y='moving_avg', label='7-day Moving Average', color='orange')
    
    plt.title("Bitcoin Price Analysis")
    plt.xlabel("Date")
    plt.ylabel("Price (USD)")
    plt.legend(loc='upper left')
    plt.grid(True)
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def main():
    price_data = []  # Store price data as a list of dictionaries for processing
    
    while True:
        price = fetch_bitcoin_price()
        
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        table = store_bitcoin_price(price, timestamp)
        
        df = etl.todataframe(table)
        
        price_data.append(df)
        
        all_data = pd.concat(price_data, ignore_index=True)
        
        all_data = time_series_analysis(all_data)
        
        visualize_data(all_data)
        
        time.sleep(10)  
if __name__ == "__main__":
    main()